# 02: Personalized Federated Learning with FedBN & Secure Aggregation

This notebook demonstrates:
1. SmallCNN model with GroupNorm (compatible with DP-SGD and FedBN).
2. Separating shared parameters (Conv, Linear) from local parameters (GroupNorm affine).
3. Pairwise Additive Masking (Bonawitz et al.) for cryptographic privacy.
4. Running federated rounds.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import torch
import yaml
from src.model import get_model
from src.fedbn import extract_shared_parameters, extract_local_parameters
from src.secure_aggregation import create_pairwise_masks_for_clients, verify_mask_cancellation
from src.federated_trainer import FederatedTrainer


In [ ]:
model = get_model(num_classes=5)
shared = extract_shared_parameters(model)
local = extract_local_parameters(model)
print(f'Shared layers count: {len(shared)}')
print(f'Local (GroupNorm) layers count: {len(local)}')


In [ ]:
client_ids = [f'hospital_{i+1}' for i in range(5)]
masks = create_pairwise_masks_for_clients(client_ids, shared, round_idx=1)
is_exact, residual = verify_mask_cancellation(masks)
print(f'Exact pairwise mask cancellation: {is_exact} (Max residual: {residual:.2e})')


In [ ]:
with open('../configs/default.yaml', 'r') as f:
    config = yaml.safe_load(f)
config['data']['csv_path'] = '../data/aptos/train.csv'
config['data']['img_dir'] = '../data/aptos/train_images'
config['output']['artifacts_dir'] = '../artifacts'
config['output']['results_dir'] = '../results'

trainer = FederatedTrainer(config=config)
print('Federated Trainer initialized successfully!')
